# Chuẩn bị dữ liệu

Tạo ảnh định danh trung tính, tập đánh giá, bảng định tuyến và nhãn chấm điểm được lưu riêng.

In [ ]:
import os
import sys
from pathlib import Path

source_root = Path(os.environ.get("MEDVQA_SOURCE_ROOT", ".")).resolve()
if not (source_root / "medvqa").is_dir():
    raise FileNotFoundError("Set MEDVQA_SOURCE_ROOT to the extracted source directory.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

## Cấu hình thí nghiệm

Chọn nguồn dữ liệu và thư mục lưu ảnh, câu hỏi, metadata định tuyến và nhãn chấm. Nhãn chấm được lưu riêng trong `SEALED_DIR`.

In [ ]:
import json
from pathlib import Path

dataset_pins = json.loads(
    (source_root / "configs/datasets.json").read_text(encoding="utf-8")
)


STAGE = "AB"
INPUT_ROOT = Path("/kaggle/input/medvqa-inputs")
VQARAD_DIR = None
SLAKE_SOURCE, SLAKE_REPO, SLAKE_REVISION = (
    "hf",
    dataset_pins["slake"]["repository"],
    dataset_pins["slake"]["revision"],
)
OMNI_SOURCE, OMNI_REPO = ("hf", dataset_pins["omnimedvqa"]["repository"])
OMNI_REVISION = dataset_pins["omnimedvqa"]["revision"]
SLAKE_LOCAL_DIR, OMNI_LOCAL_ROOT = (None, None)
RAW_DIR = Path("/tmp/mvqa_raw")
OUT_DIR = Path("/kaggle/working/prep_out")
SEALED_DIR = Path("/kaggle/working/prep_sealed")
NATIVE_SPEC = None
ROUTE_SUPPLEMENT = None
PROMPTS_DIR = None
CODE_DIR = None
NATIVE_PER_GROUP = 100
MODULE_DIR = source_root

## Thực thi và lưu kết quả

In [ ]:
import json
import re
import sys
import zipfile
from collections import Counter, defaultdict

sys.path.insert(0, str(MODULE_DIR))
VQARAD_DIR = Path(VQARAD_DIR or INPUT_ROOT / "vqarad_original")
PROMPTS_DIR = Path(PROMPTS_DIR or INPUT_ROOT / "resources/protocol" / "prompts")
CODE_DIR = Path(CODE_DIR or INPUT_ROOT / ".")
NATIVE_SPEC = Path(NATIVE_SPEC or PROMPTS_DIR / "specs" / "native_provenance.json")

In [ ]:
from medvqa import common as U
from medvqa import data_preparation as PR
from medvqa import native_data as NV
from medvqa import prompts as P
from medvqa import routing as R

SEED = U.SEED
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEALED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS = OUT_DIR / "reports"
REPORTS.mkdir(exist_ok=True)
ENV = U.environment_info()
print("Stage", STAGE, "| python", ENV["python"])

In [ ]:
def safe_extract(zf, member, root):
    target = (root / member).resolve()
    if not str(target).startswith(str(root.resolve())) or "\\" in member:
        raise ValueError(f"unsafe archive member {member}")
    zf.extract(member, root)


if "A" in STAGE:
    sources = {
        "slake": {"repo": SLAKE_REPO, "source": SLAKE_SOURCE},
        "omni": {"repo": OMNI_REPO, "source": OMNI_SOURCE},
        "vqarad": {"dir": str(VQARAD_DIR)},
    }
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    if SLAKE_SOURCE == "hf":
        from huggingface_hub import HfApi, snapshot_download

        info = HfApi().dataset_info(SLAKE_REPO, revision=SLAKE_REVISION)
        sources["slake"]["resolved_revision"] = info.sha
        SLAKE_DIR = Path(
            snapshot_download(
                SLAKE_REPO,
                repo_type="dataset",
                revision=info.sha,
                allow_patterns=["*.json", "imgs.zip"],
                local_dir=str(RAW_DIR / "slake"),
            )
        )
        with zipfile.ZipFile(SLAKE_DIR / "imgs.zip") as zf:
            for m in zf.namelist():
                safe_extract(zf, m, SLAKE_DIR)
    else:
        SLAKE_DIR = Path(SLAKE_LOCAL_DIR)
    native_ref = U.read_json(CODE_DIR / "prep" / "ref" / "native300_ref.json")
    candidates = U.read_json(
        INPUT_ROOT / "resources/inventory" / "native_heldout_candidates.json"
    )
    needed_paths = (
        {c["source_image_path"] for g in candidates["pools"].values() for c in g}
        | {x["source_image_path"] for x in native_ref["native300"]}
        | set(native_ref["legacy_paths"])
    )
    if OMNI_SOURCE == "hf":
        from huggingface_hub import hf_hub_download

        if OMNI_REVISION != native_ref["omni_revision"]:
            raise ValueError(
                "OmniMedVQA revision differs from the native300 selection contract"
            )
        zpath = hf_hub_download(
            OMNI_REPO, "OmniMedVQA.zip", repo_type="dataset", revision=OMNI_REVISION
        )
        sources["omni"]["revision"] = OMNI_REVISION
        sources["omni"]["zip_sha256"] = U.sha256_file(zpath)
        OMNI_ROOT = RAW_DIR / "omni"
        with zipfile.ZipFile(zpath) as zf:
            for m in zf.namelist():
                rel = m.split("OmniMedVQA/", 1)[-1]
                if (
                    "QA_information/Open-access/" in m
                    and m.endswith(".json")
                    or rel in needed_paths
                ):
                    safe_extract(zf, m, OMNI_ROOT)
    else:
        OMNI_ROOT = Path(OMNI_LOCAL_ROOT)
    U.write_json(REPORTS / "sources.json", sources)
    print("sources ready")

In [ ]:
if "A" in STAGE:
    slake = PR.load_slake(SLAKE_DIR)
    vq_rows = U.read_json(VQARAD_DIR / "VQA_RAD_Dataset_Public_original.json")
    VQ_IMG = VQARAD_DIR / "original_public_vqarad_images"
    split_inv = U.read_json(INPUT_ROOT / "resources/inventory" / "split_inventory.json")
    vq_split = defaultdict(set)
    for r in vq_rows:
        vq_split[r["image_name"]].add(
            "test" if r["phrase_type"].startswith("test") else "train"
        )
    vq_test_imgs = sorted((k for k, v in vq_split.items() if "test" in v))
    vq_dev_imgs = sorted((k for k, v in vq_split.items() if v == {"train"}))
    assert vq_dev_imgs == sorted(split_inv["vqarad"]["train_only_image_names"]), (
        "train-only list differs from split_inventory.json"
    )
    print(
        "SLAKE EN questions:",
        {k: len(v) for k, v in slake.items()},
        "| VQA-RAD test/dev images:",
        len(vq_test_imgs),
        len(vq_dev_imgs),
    )
    images = {}
    broken = []

    def add_image(dataset, split, name, path, is_test, extra=None):
        rgb, h = PR.decode_image(path, retries=1 if is_test else 0)
        info = {
            "dataset": dataset,
            "split": split,
            "original_name": name,
            "path": str(path),
            "rgb": h if rgb is not None else None,
            "size": list(rgb.size) if rgb is not None else None,
            "broken": rgb is None,
            **(extra or {}),
        }
        if rgb is None:
            broken.append({"dataset": dataset, "split": split, "is_test": is_test})
        images[dataset, split, name] = info
        return info

    for split in ("test", "validation"):
        for name in sorted({r["img_name"] for r in slake[split]}):
            img_id = next(
                (str(r["img_id"]) for r in slake[split] if r["img_name"] == name)
            )
            add_image(
                "SLAKE",
                split,
                name,
                PR.slake_image_path(SLAKE_DIR, name),
                split == "test",
                {"img_id": img_id},
            )
    for name in vq_test_imgs:
        add_image("VQA-RAD", "test", name, VQ_IMG / name, True)
    for name in vq_dev_imgs:
        add_image("VQA-RAD", "train", name, VQ_IMG / name, False)
    print("radiology images decoded:", len(images), "| broken:", len(broken))

In [ ]:
if "A" in STAGE:
    rq1 = U.read_json(CODE_DIR / "prep" / "ref" / "rq1_cohort_ids.json")
    rgb_by_imgid = {
        v["img_id"]: v["rgb"]
        for k, v in images.items()
        if k[0] == "SLAKE" and k[1] == "test"
    }
    slake_cmp = PR.compare_slake_with_rq1(slake["test"], rq1, rgb_by_imgid)
    vq_test_rgb = {
        v["rgb"] for k, v in images.items() if k[0] == "VQA-RAD" and k[1] == "test"
    }
    slake_cmp["vqarad_test_rgb_vs_rq1"] = {
        "n_new": len(vq_test_rgb),
        "n_rq1": len(rq1["vqarad_test_image_rgb"]),
        "n_common": len(vq_test_rgb & set(rq1["vqarad_test_image_rgb"])),
        ("note"): (
            "RQ1 hashed HF re-encoded images; differences here are informative only"
        ),
    }
    U.write_json(REPORTS / "slake_test_vs_rq1.json", slake_cmp)
    print("SLAKE test vs RQ1:", slake_cmp["claim"])
    print(
        {
            k: len(v) if isinstance(v, list) else v
            for k, v in slake_cmp.items()
            if k not in ("claim", "vqarad_test_rgb_vs_rq1")
        }
    )

In [ ]:
if "A" in STAGE:
    qa_index = PR.load_omni_qa(OMNI_ROOT)
    legacy_rgb, legacy_missing = (set(), 0)
    for p in native_ref["legacy_paths"] + native_ref["pilot_paths"]:
        try:
            rgb, h = PR.decode_image(PR.omni_image_path(OMNI_ROOT, p), retries=0)
            if rgb is not None:
                legacy_rgb.add(h)
        except FileNotFoundError:
            legacy_missing += 1
    exposed = (
        {x["image_rgb"] for x in native_ref["native300"]}
        | set(native_ref["pilot_rgb"])
        | legacy_rgb
    )
    heldout, native_ledger = PR.select_native_heldout(
        candidates, qa_index, OMNI_ROOT, exposed, NATIVE_PER_GROUP
    )
    nd_items = []
    for x in native_ref["native300"]:
        path = PR.omni_image_path(OMNI_ROOT, x["source_image_path"])
        rgb, h = PR.decode_image(path, retries=0)
        if h != x["image_rgb"]:
            raise ValueError("native300 image RGB differs from the frozen ledger")
        nd_items.append(
            {
                "candidate": x,
                "path": str(path),
                "rgb": h,
                "size": list(rgb.size),
                "item": PR.omni_item(qa_index[x["question_id"]]),
            }
        )
    U.write_json(
        SEALED_DIR / "native_selection_ledger.json",
        {"events": native_ledger, "legacy_images_missing": legacy_missing},
    )
    rep = Counter(((e["group"], e["decision"]) for e in native_ledger))
    reserve_used = {
        g: sum(
            (
                1
                for e in native_ledger
                if e["group"] == g
                and e["decision"] == "ACCEPTED"
                and (e["pool_position"] >= NATIVE_PER_GROUP)
            )
        )
        for g in PR.NATIVE_GROUPS
    }
    U.write_json(
        REPORTS / "native_heldout_selection.json",
        {
            "counts": {f"{g}|{d}": n for (g, d), n in rep.items()},
            "reserve_positions_used": reserve_used,
            "legacy_hashed": len(legacy_rgb),
            "legacy_missing": legacy_missing,
        },
    )
    print(
        "native-heldout:",
        {g: len(v) for g, v in heldout.items()},
        "| decisions:",
        dict(Counter((e["decision"] for e in native_ledger))),
    )

In [ ]:
if "A" in STAGE:
    test_rgb = {v["rgb"] for k, v in images.items() if k[1] == "test" and v["rgb"]} | {
        x["rgb"] for g in heldout.values() for x in g
    }
    dedup = {
        "vqarad_dev_removed": [],
        "slake_val_equal_test": 0,
        "rad_test_duplicate_groups": 0,
        "dev_broken_removed": 0,
    }
    for k in list(images):
        v = images[k]
        if k[0] == "VQA-RAD" and k[1] == "train" and (v["rgb"] in test_rgb):
            dedup["vqarad_dev_removed"].append(k[2])
            del images[k]
        elif k[1] in ("train", "validation") and v["broken"]:
            dedup["dev_broken_removed"] += 1
            del images[k]
        elif k[0] == "SLAKE" and k[1] == "validation" and (v["rgb"] in test_rgb):
            dedup["slake_val_equal_test"] += 1
            v["equals_test_rgb"] = True
    rt_groups = Counter(
        (v["rgb"] for k, v in images.items() if k[1] == "test" and v["rgb"])
    )
    dedup["rad_test_duplicate_groups"] = sum((1 for n in rt_groups.values() if n > 1))
    U.write_json(SEALED_DIR / "dedup_names.json", dedup)
    U.write_json(
        REPORTS / "dedup.json",
        {k: len(v) if isinstance(v, list) else v for k, v in dedup.items()},
    )
    print("dedup:", {k: len(v) if isinstance(v, list) else v for k, v in dedup.items()})

In [ ]:
if "A" in STAGE:
    groups = {"RT": [], "RD": [], "NT": [], "ND": []}
    for k, v in images.items():
        groups["RT" if k[1] == "test" else "RD"].append(v)
    for g, lst in heldout.items():
        for x in lst:
            groups["NT"].append(
                {
                    "dataset": "OmniMedVQA",
                    "split": "heldout",
                    "original_name": x["candidate"]["source_image_path"],
                    "path": x["path"],
                    "rgb": x["rgb"],
                    "size": x["size"],
                    "broken": False,
                    "native_group": g,
                    "omni": x["item"],
                    "omni_qid": x["candidate"]["question_id"],
                    "source_dataset": x["candidate"].get("source_dataset"),
                }
            )
    for x in nd_items:
        groups["ND"].append(
            {
                "dataset": "OmniMedVQA",
                "split": "native300",
                "original_name": x["candidate"]["source_image_path"],
                "path": x["path"],
                "rgb": x["rgb"],
                "size": x["size"],
                "broken": False,
                "native_group": x["candidate"]["modality"],
                "omni": x["item"],
                "omni_qid": x["candidate"]["question_id"],
                "source_dataset": x["candidate"].get("source_dataset"),
            }
        )
    for pfx in groups:
        groups[pfx] = PR.assign_neutral_ids(pfx, groups[pfx])
    IMG_DIR = OUT_DIR / "images"
    from PIL import Image

    native = NV.NativeAssigner(NATIVE_SPEC)
    native_records, native_ds = ({}, {})
    index = {}
    for pfx, lst in groups.items():
        for v in lst:
            if not v["broken"]:
                with Image.open(v["path"]) as im:
                    rgb_im = im.convert("RGB")
                PR.save_png(rgb_im, IMG_DIR / f"{v['neutral_id']}.png")
                if pfx in ("NT", "ND"):
                    with Image.open(IMG_DIR / f"{v['neutral_id']}.png") as png:
                        rec, ds = native.record(
                            v["neutral_id"],
                            v["original_name"],
                            v.get("source_dataset"),
                            png.convert("RGB"),
                        )
                    native_records[v["neutral_id"]], native_ds[v["neutral_id"]] = (
                        rec,
                        ds,
                    )
            index[v["neutral_id"]] = {
                "prefix": pfx,
                "cluster_id": v["rgb"],
                "size": v["size"],
                "broken": v["broken"],
                "equals_test_rgb": bool(v.get("equals_test_rgb")),
            }
    U.write_json(OUT_DIR / "images_index.json", index)
    U.write_json(
        OUT_DIR / "run_config_prep.json",
        {
            "notebook": "prep_data",
            "native_spec": "NATIVE-PROVENANCE-v1",
            "native_spec_sha256": U.sha256_file(NATIVE_SPEC),
            "image_decoder": NV.decoder_versions(),
            "env": U.environment_info(),
        },
    )
    print(
        {p: len(line_value) for p, line_value in groups.items()},
        "PNG written:",
        len(list(IMG_DIR.glob("*.png"))),
    )

In [ ]:
if "A" in STAGE:
    routes_spec_a = R.load_routes_spec(PROMPTS_DIR)
    native_pairs = R.build_route_pairs(native_records, routes_spec_a)
    native_route = {
        n: R.route_for(n, r, native_pairs)["route_key"]
        for n, r in native_records.items()
    }
    cohort_name = {"NT": "COH-NAT-TEST", "ND": "COH-NAT-DEV"}
    agg = {}
    for n, rec in native_records.items():
        key = f"{cohort_name[n[:2]]}|{native_ds[n]}"
        a = agg.setdefault(
            key, {"n": 0, "route": Counter(), "route_reason": Counter(), "ck_fail": 0}
        )
        a["n"] += 1
        a["route"][native_route[n]] += 1
        a["route_reason"][rec["route_reason"]] += 1
        a["ck_fail"] += rec["route_reason"] in ("ck_color", "ck_gray")
    native_counts = {
        k: {
            "n": v["n"],
            "route": dict(v["route"]),
            "route_reason": dict(v["route_reason"]),
            "ck_fail": v["ck_fail"],
        }
        for k, v in sorted(agg.items())
    }
    mass_table, mass_bad = NV.mass_fail_check(
        [
            (native_ds[n], r["route_reason"])
            for n, r in native_records.items()
            if n.startswith("ND")
        ]
    )
    U.write_json(
        REPORTS / "native_provenance_counts.json",
        {
            "by_cohort_dataset": native_counts,
            "ck_color_dev_by_dataset": mass_table,
            "mass_fail_datasets": mass_bad,
        },
    )
    del native_ds
    print(
        "native route reasons:",
        dict(Counter((r["route_reason"] for r in native_records.values()))),
    )
    if mass_bad:
        raise NV.NativeBuildStop(
            (
                "CK-COLOR mass-fail rule: "
                f"{len(mass_bad)}"
                " dataset(s) with > 20% fails in COH-NAT-DEV. "
                "Routing metadata validation failed; "
                "thresholds must not be tuned."
            )
        )

In [ ]:
if "A" in STAGE:
    cohort_rows = {
        "COH-RAD-TEST": [],
        "COH-NAT-TEST": [],
        "COH-RAD-DEV": [],
        "COH-NAT-DEV": [],
    }
    gold = {"RAD_TEST": [], "NAT_TEST": [], "DEV": []}
    id_map = {}
    slake_by_img = {s: defaultdict(list) for s in ("test", "validation")}
    for s in ("test", "validation"):
        for r in slake[s]:
            slake_by_img[s][r["img_name"]].append(r)
    vq_by_img = defaultdict(list)
    for r in vq_rows:
        vq_by_img[r["image_name"]].append(r)
    route_inputs = {"slake": {}, "vqarad": {}, "native": native_records}
    vq_inv = {
        r["image_name"]: r["modality_visual"]
        for r in U.read_json(
            INPUT_ROOT
            / "resources/research"
            / "clinical"
            / "ct"
            / "vqarad_modality_inventory.json"
        )["rows"]
    }
    vq_dev_inv_doc = U.read_json(
        INPUT_ROOT
        / "resources/research"
        / "clinical"
        / "vqarad_dev_modality_inventory.json"
    )
    vq_dev_r2 = U.read_json(
        INPUT_ROOT / "resources/annotations" / "vqarad_dev_modality_rater2.json"
    )["rows"]
    vq_dev_inv, r2_report = ({}, {"agree": 0, "disagree": 0, "missing_rater2": 0})
    for r in vq_dev_inv_doc["rows"]:
        m1, m2 = (
            r["modality_visual"],
            (vq_dev_r2.get(r["image_name"]) or {}).get("modality"),
        )
        if m2 is None:
            r2_report["missing_rater2"] += 1
            vq_dev_inv[r["image_name"]] = "unknown"
        elif m1 == m2:
            r2_report["agree"] += 1
            vq_dev_inv[r["image_name"]] = m1
        else:
            r2_report["disagree"] += 1
            vq_dev_inv[r["image_name"]] = "conflict"
    missing_dev = sorted(set(vq_dev_imgs) - set(vq_dev_inv))
    if missing_dev:
        raise ValueError(
            f"{len(missing_dev)} VQA-RAD dev images absent from the two-rater inventory"
        )
    U.write_json(REPORTS / "vqarad_dev_two_rater_check.json", r2_report)
    print("VQA-RAD dev modality, rater1 vs rater2:", r2_report)

    def emit(pfx, v, qas):
        nid = v["neutral_id"]
        cohort = {
            "RT": "COH-RAD-TEST",
            "NT": "COH-NAT-TEST",
            "RD": "COH-RAD-DEV",
            "ND": "COH-NAT-DEV",
        }[pfx]
        gkey = {"RT": "RAD_TEST", "NT": "NAT_TEST"}.get(pfx, "DEV")
        id_map[nid] = {
            "dataset": v["dataset"],
            "split": v["split"],
            "original_name": v["original_name"],
            "img_id": v.get("img_id"),
            "questions": {},
        }
        for k, q in enumerate(qas, 1):
            qid = f"{nid}-Q{k:02d}"
            row = {
                "qid": qid,
                "neutral_id": nid,
                "cohort": cohort,
                "question": q["question"],
                "answer_type_official": q["atype"],
                "answer_kind": q["kind"],
                "options": q.get("options"),
                "K": q.get("K"),
                "orders": q.get("orders"),
                "cluster_id": v["rgb"],
                "image_broken": v["broken"],
                "strata": q["strata"],
            }
            cohort_rows[cohort].append(row)
            gold[gkey].append(
                {
                    "qid": qid,
                    "gold_answer": q["gold"],
                    "gold_label": q.get("gold_label"),
                }
            )
            id_map[nid]["questions"][qid] = q["orig_qid"]

    for pfx in ("RT", "RD"):
        for v in groups[pfx]:
            if v["dataset"] == "SLAKE":
                rows = slake_by_img[v["split"]][v["original_name"]]
                qas = [
                    {
                        "question": r["question"],
                        "atype": str(r["answer_type"]).strip().upper(),
                        "kind": P.answer_kind(r["answer_type"], r["question"]),
                        "gold": str(r["answer"]),
                        "orig_qid": str(r["qid"]),
                        "strata": {
                            "dataset": "SLAKE",
                            "slake_content_type": r.get("content_type"),
                            "native_group": None,
                        },
                    }
                    for r in rows
                ]
                route_inputs["slake"][v["neutral_id"]] = [
                    {"modality": r["modality"], "location": r["location"]} for r in rows
                ]
            else:
                rows = [
                    r
                    for r in vq_by_img[v["original_name"]]
                    if r["phrase_type"].startswith("test") == (v["split"] == "test")
                ]
                qas = [
                    {
                        "question": r["question"],
                        "atype": str(r["answer_type"]).strip().upper(),
                        "kind": P.answer_kind(r["answer_type"], r["question"]),
                        "gold": str(r["answer"]),
                        "orig_qid": str(r["qid"]),
                        "strata": {
                            "dataset": "VQA-RAD",
                            "vqarad_image_organ": r["image_organ"].strip(),
                            "vqarad_question_type": r["question_type"],
                            "native_group": None,
                        },
                    }
                    for r in rows
                ]
                if v["split"] == "test":
                    mod, msrc = (
                        vq_inv.get(v["original_name"]),
                        "ct/vqarad_modality_inventory.json (P0, 2 raters)",
                    )
                else:
                    mod, msrc = (
                        vq_dev_inv.get(v["original_name"]),
                        "vqarad_dev_modality_inventory.json (2 raters, kappa 1.00)",
                    )
                route_inputs["vqarad"][v["neutral_id"]] = {
                    "organs": [r["image_organ"].strip() for r in rows],
                    "modality": mod or "unknown",
                    "modality_source": msrc,
                    "extra_gaps": PR.VQARAD_EXTRA_GAPS.get(v["original_name"], []),
                }
            emit(pfx, v, qas)
    for pfx in ("NT", "ND"):
        for v in groups[pfx]:
            it = v["omni"]
            K = len(it["options"])
            qas = [
                {
                    "question": it["question"],
                    "atype": "MCQ",
                    "kind": "mcq",
                    "options": [
                        {"label": "ABCD"[i], "text": t}
                        for i, t in enumerate(it["options"])
                    ],
                    "K": K,
                    "orders": list(range(K)),
                    "gold": it["gold_answer"],
                    "gold_label": it["gold_label"],
                    "orig_qid": v["omni_qid"],
                    "strata": {
                        "dataset": "OmniMedVQA",
                        "native_group": v["native_group"],
                        "omni_question_type": it["question_type"],
                    },
                }
            ]
            emit(pfx, v, qas)
    for c, rows in cohort_rows.items():
        p = OUT_DIR / "cohorts" / f"{c}.jsonl"
        p.parent.mkdir(exist_ok=True)
        p.write_text(
            "".join((json.dumps(r, ensure_ascii=False) + "\n" for r in rows)),
            encoding="utf-8",
        )
    for g, rows in gold.items():
        (SEALED_DIR / f"gold_{g}.jsonl").write_text(
            "".join((json.dumps(r, ensure_ascii=False) + "\n" for r in rows)),
            encoding="utf-8",
        )
    U.write_json(SEALED_DIR / "id_map.json", id_map)
    U.write_json(
        SEALED_DIR / "original_names.json",
        sorted(
            {Path(v["original_name"]).name for v in id_map.values()}
            | {v["original_name"] for v in id_map.values()}
        ),
    )
    U.write_json(OUT_DIR / "route_inputs" / "route_inputs.json", route_inputs)
    _broken = [v["neutral_id"] for v in groups["RT"] if v["broken"]]
    U.write_json(
        REPORTS / "broken_images.json",
        {"neutral_ids": _broken, "radiology_test_broken_neutral_ids": _broken},
    )
    summary = {
        c: {
            "questions": len(r),
            "images": len({x["neutral_id"] for x in r}),
            "kinds": dict(Counter((x["answer_kind"] for x in r))),
            "closed_official": sum((x["answer_type_official"] == "CLOSED" for x in r)),
        }
        for c, r in cohort_rows.items()
    }
    U.write_json(REPORTS / "cohort_summary.json", summary)
    for c, s in summary.items():
        print(c, s)

In [ ]:
if "A" in STAGE:
    from PIL import Image

    def annotate(prefix):
        ann = {}
        for v in groups[prefix]:
            if v["dataset"] != "SLAKE" or v["broken"]:
                continue
            folder = Path(v["path"]).parent
            det = (
                json.loads((folder / "detection.json").read_text(encoding="utf-8"))
                if (folder / "detection.json").is_file()
                else []
            )
            W, H = v["size"]
            if (folder / "mask.png").is_file():
                with Image.open(folder / "mask.png") as m:
                    if m.size != (W, H):
                        raise ValueError("mask geometry differs from image")
            entries = PR.annotation_entries(det, W, H)
            if entries:
                ann[v["neutral_id"]] = {
                    "entries": entries,
                    "text": PR.annotation_text(entries, W, H),
                    "size": [W, H],
                }
        return ann

    ann = annotate("RT")
    U.write_json(OUT_DIR / "judge" / "annotations_slake_test.json", ann)
    probes = PR.build_probes(ann)
    U.write_json(OUT_DIR / "judge" / "jprobe_set.json", probes)
    ann_dev = annotate("RD")
    U.write_json(OUT_DIR / "judge" / "annotations_slake_dev.json", ann_dev)
    probes_dev = PR.build_probes(ann_dev)
    U.write_json(OUT_DIR / "judge" / "jprobe_set_dev.json", probes_dev)
    print(
        "annotated test images:",
        len(ann),
        "| probes:",
        probes["counts"],
        "| dev images:",
        len(ann_dev),
        "| dev probes:",
        probes_dev["counts"],
    )
    print(
        (
            "Stage A done. Download prep_out/ and "
            "prep_sealed/ separately (runs/prep/). Do NOT "
            "upload prep_sealed/ for GPU notebooks."
        )
    )

In [ ]:
if "B" in STAGE:
    route_inputs = U.read_json(OUT_DIR / "route_inputs" / "route_inputs.json")
    records, nonconstant = ({}, [])
    for nid, rows in route_inputs["slake"].items():
        records[nid] = R.slake_record(rows)
    for nid, x in route_inputs["vqarad"].items():
        records[nid] = R.vqarad_record(
            x["organs"], x["modality"], x["modality_source"], x.get("extra_gaps", [])
        )
    for nid, rec in route_inputs["native"].items():
        NV.validate_record(rec)
        records[nid] = rec
    nonconstant = sorted(
        (n for n, r in records.items() if r.get("status") == "nonconstant")
    )
    print(
        "records:",
        len(records),
        "| status:",
        dict(Counter((R.view(r)["status"] for r in records.values()))),
        "| nonconstant (check 9):",
        len(nonconstant),
    )

In [ ]:
if "B" in STAGE:
    routes_spec = R.load_routes_spec(PROMPTS_DIR)
    supplement = U.read_json(ROUTE_SUPPLEMENT) if ROUTE_SUPPLEMENT else None
    route_pairs = R.build_route_pairs(records, routes_spec, supplement)
    U.write_json(OUT_DIR / "route_pairs.json", route_pairs)
    routing = {n: R.route_for(n, records[n], route_pairs) for n in sorted(records)}
    weak_routes = {r for r, v in routes_spec["routes"].items() if v.get("weak_group")}
    for n, rec in records.items():
        if "provenance_rule" in rec:
            continue
        slake_mr_chest = (
            rec["source_kind"] == "SLAKE"
            and rec["modality"] == "MRI"
            and (rec.get("regions") in (["Lung"], ["Chest"]))
        )
        rec["weak_group"] = (
            bool(set(routing[n]["routes"]) & weak_routes) or slake_mr_chest
        )
    U.write_json(
        OUT_DIR / "route_metadata.json",
        {"version": "ROUTE-METADATA-v1", "records": records},
    )
    index = U.read_json(OUT_DIR / "images_index.json")

    def cohort_of(n):
        return {
            "RT": "COH-RAD-TEST",
            "NT": "COH-NAT-TEST",
            "RD": "COH-RAD-DEV",
            "ND": "COH-NAT-DEV",
        }[n[:2]]

    fb = Counter(
        (cohort_of(n) for n, r in routing.items() if r["route_key"] == "RT-FALLBACK")
    )
    rc = Counter(((cohort_of(n), r["route_key"]) for n, r in routing.items()))
    fb_reason = Counter(
        (
            (cohort_of(n), r["fallback_reason"])
            for n, r in routing.items()
            if r["route_key"] == "RT-FALLBACK"
        )
    )
    U.write_json(
        REPORTS / "routing_summary.json",
        {
            "fallback_by_cohort": dict(fb),
            "nonconstant_check9": nonconstant,
            "fallback_by_cohort_reason": {
                f"{c}|{k}": v for (c, k), v in sorted(fb_reason.items())
            },
            "route_counts": {f"{c}|{k}": v for (c, k), v in sorted(rc.items())},
        },
    )
    print("pairs:", len(route_pairs["pairs"]), "| FALLBACK by cohort:", dict(fb))

In [ ]:
if "B" in STAGE:

    def load_cohort(name):
        return [
            json.loads(line_value)
            for line_value in (OUT_DIR / "cohorts" / f"{name}.jsonl")
            .read_text(encoding="utf-8")
            .splitlines()
            if line_value.strip()
        ]

    rad_dev, nat_dev = (load_cohort("COH-RAD-DEV"), load_cohort("COH-NAT-DEV"))
    ds_of = {r["neutral_id"]: r["strata"]["dataset"] for r in rad_dev}
    grp_of = {r["neutral_id"]: r["strata"]["native_group"] for r in nat_dev}
    rk = {n: routing[n]["route_key"] for n in routing}
    pilot = []
    for ds in ("SLAKE", "VQA-RAD"):
        ids = sorted(
            (
                n
                for n, d in ds_of.items()
                if d == ds
                and (not index[n]["broken"])
                and (not index[n].get("equals_test_rgb"))
            )
        )
        pilot += PR.stratified_pick(ids, rk, 20, f"PILOT|{ds}")
    for g in PR.NATIVE_GROUPS:
        ids = sorted((n for n, x in grp_of.items() if x == g))
        pilot += PR.stratified_pick(ids, rk, 10, f"PILOT|{g}")
    pset = set(pilot)
    pilot_rows = [
        dict(r, cohort_source=r["cohort"], cohort="COH-PILOT")
        for r in rad_dev + nat_dev
        if r["neutral_id"] in pset
    ]
    (OUT_DIR / "cohorts" / "COH-PILOT.jsonl").write_text(
        "".join((json.dumps(r, ensure_ascii=False) + "\n" for r in pilot_rows)),
        encoding="utf-8",
    )
    print("COH-PILOT images:", len(pset), "| questions:", len(pilot_rows))

In [ ]:
if "B" in STAGE:
    rad_test, nat_test = (load_cohort("COH-RAD-TEST"), load_cohort("COH-NAT-TEST"))
    slake_t = sorted(
        {r["neutral_id"] for r in rad_test if r["strata"]["dataset"] == "SLAKE"}
    )
    organ = {
        r["neutral_id"]: r["strata"]["vqarad_image_organ"]
        for r in rad_test
        if r["strata"]["dataset"] == "VQA-RAD"
    }
    grp = {r["neutral_id"]: r["strata"]["native_group"] for r in nat_test}
    panel = {"SLAKE": slake_t}
    for o in ("ABD", "HEAD", "CHEST"):
        panel[f"VQA-RAD|{o}"] = PR.first_by_hash(
            [n for n, x in organ.items() if x == o], 20, f"PANEL|{o}"
        )
    for g in PR.NATIVE_GROUPS:
        panel[f"NAT|{g}"] = PR.first_by_hash(
            [n for n, x in grp.items() if x == g], 20, f"PANEL|{g}"
        )
    U.write_json(
        OUT_DIR / "panel_rq2.json",
        {
            "version": "PANEL-RQ2-v1",
            "seed": SEED,
            "strata": panel,
            "neutral_ids": sorted({n for v in panel.values() for n in v}),
        },
    )
    print("panel:", {k: len(v) for k, v in panel.items()})

In [ ]:
if "B" in STAGE:
    from medvqa import interventions as I

    pilot_rows = load_cohort("COH-PILOT")
    shuf = {}
    for mode_name, rr in (("TEST", rad_test + nat_test), ("PILOT", pilot_rows)):
        ids = sorted({r["neutral_id"] for r in rr})
        coh = {r["neutral_id"]: r["cohort"] for r in rr}
        shuf[mode_name] = I.shuf_table(
            ids,
            {n: routing[n]["route_key"] for n in ids},
            {n: R.modality_family(routing[n]["route_key"], records[n]) for n in ids},
            coh,
        )
    U.write_json(
        OUT_DIR / "shuf_donors.json", {"version": "SHUF-DONORS-v2", "by_mode": shuf}
    )
    print(
        "SHUF tiers:",
        {m: t["tier_counts"] for m, t in shuf.items()},
        "| int_missing:",
        {m: len(t["int_missing"]) for m, t in shuf.items()},
    )

In [ ]:
if "B" in STAGE:
    bank = P.ObserverPromptBank(PROMPTS_DIR)
    all_rows = rad_test + nat_test + rad_dev + nat_dev
    questions = sorted({r["question"] for r in all_rows})
    options = sorted(
        {o["text"] for r in all_rows if r["options"] for o in r["options"]}
    )
    golds = sorted(
        {
            g["gold_answer"]
            for f in ("RAD_TEST", "NAT_TEST", "DEV")
            for g in U.read_jsonl(SEALED_DIR / f"gold_{f}.jsonl")
        }
    )
    names = U.read_json(SEALED_DIR / "original_names.json")
    manifest, leak_report = PR.observer_prompt_manifest(
        bank, routing, questions, options, golds, names
    )
    del golds, names
    U.write_json(OUT_DIR / "observer_prompt_manifest.json", manifest)
    U.write_json(REPORTS / "leakage_checks_prep.json", leak_report)
    print("observer prompts:", len(manifest["entries"]), "|", leak_report)

In [ ]:
if "B" in STAGE:
    locked = [
        "route_metadata.json",
        "route_pairs.json",
        "observer_prompt_manifest.json",
        "panel_rq2.json",
        "shuf_donors.json",
        "cohorts/COH-RAD-TEST.jsonl",
        "cohorts/COH-NAT-TEST.jsonl",
        "cohorts/COH-RAD-DEV.jsonl",
        "cohorts/COH-NAT-DEV.jsonl",
        "cohorts/COH-PILOT.jsonl",
        "images_index.json",
        "judge/annotations_slake_test.json",
        "judge/jprobe_set.json",
        "judge/annotations_slake_dev.json",
        "judge/jprobe_set_dev.json",
    ]
    lines = [f"{U.sha256_file(OUT_DIR / p)}  prep_out/{p}" for p in locked]

    def rd(f):
        return U.read_json(REPORTS / f)

    sel, slk, nat, rsum = (
        rd("native_heldout_selection.json"),
        rd("slake_test_vs_rq1.json"),
        rd("native_provenance_counts.json"),
        rd("routing_summary.json"),
    )
    eda = {
        "version": "EDA-SUMMARY-v1",
        "images": {
            "dedup": rd("dedup.json"),
            "radiology_test_broken": len(rd("broken_images.json")["neutral_ids"]),
            "native_selection_decisions": sel["counts"],
            "native_reserve_positions_used": sel["reserve_positions_used"],
            "legacy_images_hashed": sel.get("legacy_hashed"),
            "legacy_images_missing": sel.get("legacy_missing"),
        },
        "cohorts": rd("cohort_summary.json"),
        "slake_test_vs_rq1": {
            "identical": slk["identical"],
            "claim": slk["claim"],
            **{
                k: len(v) if isinstance(v, list) else v
                for k, v in slk.items()
                if k not in ("identical", "claim")
            },
        },
        "native_by_dataset": nat["by_cohort_dataset"],
        "ck_color_dev_by_dataset": nat["ck_color_dev_by_dataset"],
        "fallback_by_cohort_reason": rsum["fallback_by_cohort_reason"],
        "route_by_cohort": rsum["route_counts"],
        "nonconstant_check9": len(rsum["nonconstant_check9"]),
        "shuf_donors": {
            m: {
                "tier_counts": t["tier_counts"],
                "int_missing_images": len(t["int_missing"]),
            }
            for m, t in shuf.items()
        },
    }
    blob = json.dumps(eda, ensure_ascii=False)
    if (
        re.search("\\b(RT|NT|RD|ND)-\\d{4}\\b", blob)
        or "Images/" in blob
        or re.search("\\.(png|jpe?g)\\b|synpic|xmlab", blob, re.I)
    ):
        raise U.IntegrityError(
            (
                "eda_summary.json must not contain neutral_ids, "
                "paths or original image names"
            )
        )
    U.write_json(REPORTS / "eda_summary.json", eda)
    (REPORTS / "locked_prep_outputs.sha256").write_text(
        "\n".join(lines) + "\n", encoding="utf-8"
    )
    print("\n".join(lines))